<a href="https://colab.research.google.com/github/rithika-d/Big_Data_Analytics_Midterm_Project/blob/main/Big_Data_Analytics_Midterm_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia/data


https://colab.research.google.com/drive/1ZZgTz6XlD_4qtLrZr-v3UJTcRrX428Z-?authuser=1

https://colab.research.google.com/drive/1SGjVnk1JAwCyquc4qsdu2rp6VoV9px5r?authuser=1

https://github.com/hustvl/EVA-X
https://arxiv.org/html/2405.05237v1

https://huggingface.co/microsoft/llava-rad

https://github.com/mbzuai-oryx/XrayGPT?tab=readme-ov-file


In [ ]:
#https://www.google.com/url?q=https%3A%2F%2Fwww.kaggle.com%2Fdatasets%2Fpaultimothymooney%2Fchest-xray-pneumonia%2Fdata

!pip install torch torchvision timm==0.9.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.3 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.25
    Uninstalling timm-1.0.25:
      Successfully uninstalled timm-1.0.25


In [ ]:
import importlib
import eva_x
importlib.reload(eva_x)
print("eva_x module reloaded successfully.")

eva_x module reloaded successfully.


In [ ]:
#https://github.com/hustvl/EVA-X?tab=readme-ov-file#quick-start
from eva_x import eva_x_tiny_patch16

model = eva_x_tiny_patch16(pretrained="/content/drive/MyDrive/chest_xray/eva_x_tiny_patch16_merged520k_mim.pt")

_IncompatibleKeys(missing_keys=['head.weight', 'head.bias'], unexpected_keys=[])


In [ ]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision

from torch.utils.data import Dataset, DataLoader, BatchSampler, random_split
from torchvision import transforms
from PIL import Image
from transformers import ViTImageProcessor

torch.backends.cudnn.benchmark = True

In [ ]:
#https://docs.pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html
#https://docs.pytorch.org/vision/0.21/transforms.html
#https://divyanshu1331.medium.com/why-vit-struggles-on-small-datasets-a-4-stage-experiment-on-cifar-10-7634ab2ae94c
#https://github.com/Divyanshu1331/cv-research-projects/blob/main/ViT_on_CIFAR10/ViT_on_CIFAR10_02.ipynb

import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. Define transformations

train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
  ])

test_val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# 2. Load the dataset
train_dataset = datasets.ImageFolder(root='/content/drive/MyDrive/chest_xray/train/', transform=train_transforms)
val_dataset = datasets.ImageFolder(root='/content/drive/MyDrive/chest_xray/val/', transform=test_val_transforms)
test_dataset = datasets.ImageFolder(root='/content/drive/MyDrive/chest_xray/test/', transform=test_val_transforms)

# 3. Create the DataLoader
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
print(train_dataset.class_to_idx)

{'NORMAL': 0, 'PNEUMONIA': 1}


In [ ]:
print(val_dataset.class_to_idx)

{'NORMAL': 0, 'PNEUMONIA': 1}


In [ ]:
print(test_dataset.class_to_idx)

{'NORMAL': 0, 'PNEUMONIA': 1}


In [ ]:
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

Train size: 3253
Val size: 16
Test size: 624


In [ ]:
img, label = train_dataset[0]
print(img.shape)
print(label)
print(img.min(), img.max())

torch.Size([3, 224, 224])
0
tensor(-2.1179) tensor(2.4308)


In [ ]:
train_dataset[0]

(tensor([[[-1.7240, -1.7412, -1.7412,  ..., -0.8164, -0.7479, -0.7137],
          [-1.7069, -1.7240, -1.7240,  ..., -0.7650, -0.7308, -0.6794],
          [-1.7240, -1.7240, -1.7412,  ..., -0.6965, -0.6623, -0.6109],
          ...,
          [-1.6384, -1.6384, -1.6042,  ...,  0.3309,  0.3994,  0.4851],
          [-1.6213, -1.6384, -1.6213,  ...,  0.3309,  0.3994,  0.4851],
          [-1.6213, -1.6384, -1.6384,  ...,  0.3481,  0.4508,  0.5022]],
 
         [[-1.6331, -1.6506, -1.6506,  ..., -0.7052, -0.6352, -0.6001],
          [-1.6155, -1.6331, -1.6331,  ..., -0.6527, -0.6176, -0.5651],
          [-1.6331, -1.6331, -1.6506,  ..., -0.5826, -0.5476, -0.4951],
          ...,
          [-1.5455, -1.5455, -1.5105,  ...,  0.4678,  0.5378,  0.6254],
          [-1.5280, -1.5455, -1.5280,  ...,  0.4678,  0.5378,  0.6254],
          [-1.5280, -1.5455, -1.5455,  ...,  0.4853,  0.5903,  0.6429]],
 
         [[-1.4036, -1.4210, -1.4210,  ..., -0.4798, -0.4101, -0.3753],
          [-1.3861, -1.4036,

In [ ]:
train_dataset.samples[0][0]

'/content/drive/MyDrive/chest_xray/train/NORMAL/IM-0115-0001.jpeg'

Input: x shaped (B, 3, 224, 224)

Output: binary_logits shaped (B, 1)

Use BCEWithLogitsLoss during training

Apply sigmoid only for inference/metrics

In [ ]:
from re import X
import timm
import torch.nn as nn
import torch.nn.functional as F

class Eva_X_Model(nn.Module):
    def __init__(self):
        super().__init__()
        'img_size = 224, patch_size = 16, embed_dim = 192, depth = 12 transformer layers, num_heads = 3'

        self.eva = eva_x_tiny_patch16(pretrained="/content/drive/MyDrive/chest_xray/eva_x_tiny_patch16_merged520k_mim.pt")
        self.eva.head = nn.Linear(192, 1) #normal vs. abnormal

        #freeze entire Eva
        for p in self.eva.parameters():
            p.requires_grad = False

        #unfreeze last transformer block + norm
        for name, p in self.eva.named_parameters():
            if "head" in name or "blocks.11" in name or "norm" in name or "fc_norm" in name:
                p.requires_grad = True

    def forward(self, x):
        binary_logits = self.eva(x) #returns (B, 1) logits

        return binary_logits

https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html

In [ ]:
from torch.cuda.amp import autocast, GradScaler

class Trainer:
    def __init__(
        self,
        model,
        criterion,
        optimizer,
        train_dataset,
        train_loader,
        val_loader,
        test_dataset,
        test_loader,
        device="cuda"
    ):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.train_dataset = train_dataset
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_dataset = test_dataset
        self.test_loader = test_loader
        self.device = device

        self.use_amp = torch.cuda.is_available()
        self.scaler = GradScaler(enabled=self.use_amp)

    def train_epoch(self):
        self.model.train()
        running_loss = 0.0

        for image, label in self.train_loader:
            image = image.to(self.device, non_blocking=True)
            label = label.to(self.device, non_blocking=True)
            #for BCEWithLogitsLoss, labels must be float and usually shape (B,1) to match logits (B,1)
            label = label.unsqueeze(1).float()

            self.optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=self.use_amp):
                logits = self.model(image)
                loss = self.criterion(logits, label)

            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()

            running_loss += loss.detach().float().item()

        print(f"Training loss: {running_loss / len(self.train_loader):.4f}")

    def validate_epoch(self):
        self.model.eval()

        total = 0
        correct = 0
        running_loss = 0.0

        with torch.no_grad():
            for image, label in self.val_loader:
                image = image.to(self.device)
                label = label.to(self.device)
                #for BCEWithLogitsLoss, labels must be float and usually shape (B,1) to match logits (B,1)
                label = label.unsqueeze(1).float()

                logits = self.model(image)

                loss = self.criterion(logits, label)
                pred = torch.sigmoid(logits)
                pred = (pred > 0.5).float()
                total += label.size(0)
                correct += (pred == label).sum().item()
                running_loss += loss.item()

        print(f"Validation loss: {running_loss / len(self.val_loader):.4f}")
        print(f"Validation acc: {100 * correct / total:.2f}%")

        return running_loss / len(self.val_loader)

    def test(self, save_to_csv=False):
        if self.test_loader is None:
            raise ValueError("test_loader not provided")

        self.model.eval()
        predictions = {
            "image_path": [],
            "y_pred": [],
            "confidence": [],
            "label": []
        }

        with torch.no_grad():
            count = 0
            for image, label in self.test_loader:
                batch_size = len(label)
                image = image.to(self.device)
                logits = self.model(image)

                y_prob = torch.sigmoid(logits) #model confidence
                y_pred = (y_prob > 0.5).int() #hard binary classification

                y_pred_list = y_pred.cpu().numpy().flatten().tolist()
                y_true = label.cpu().numpy().tolist()

                paths = self.test_dataset.samples[count:count+batch_size] #batch slice
                paths = [path[0] for path in paths] #extracts only filename paths

                predictions["image_path"].extend(paths)
                predictions["y_pred"].extend(y_pred_list)
                predictions["confidence"].extend(y_prob.cpu().numpy().flatten().tolist())
                predictions["label"].extend(y_true)
                count += batch_size

        df = pd.DataFrame(predictions)

        if save_to_csv:
            df.to_csv("/content/drive/MyDrive/chest_xray/test_predictions.csv", index=False)

        return df

In [ ]:
from collections import Counter
print(Counter(train_dataset.targets))

Counter({1: 1912, 0: 1341})


In [ ]:
import torch.optim as optim

model = Eva_X_Model()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

pos_weight = torch.tensor([0.70], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight) #1341/1912 class imbalance: Counter({1: 1912, 0: 1341})
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_dataset=train_dataset,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    test_dataset=test_dataset,
    test_loader=test_dataloader,
    device=device
)


_IncompatibleKeys(missing_keys=['head.weight', 'head.bias'], unexpected_keys=[])


/tmp/ipython-input-288/2969428516.py:27: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.use_amp)


In [ ]:
# Early stopping + save best checkpoint to Google Drive

import os
import torch

num_epochs = 20
patience = 4
min_delta = 1e-3

best_val_loss = float("inf")
epochs_no_improve = 0

# Drive checkpoint path
ckpt_dir = "/content/drive/MyDrive/chest_xray/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path = os.path.join(ckpt_dir, "eva_x_tiny_binary_best.pt")

best_state = {
    "model": {k: v.detach().cpu() for k, v in trainer.model.state_dict().items()},
    "optimizer": trainer.optimizer.state_dict(),
    "epoch": -1,
    "best_val_loss": best_val_loss,
}

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}")

    trainer.train_epoch()
    val_loss = trainer.validate_epoch()

    if val_loss < best_val_loss - min_delta:
        best_val_loss = val_loss
        epochs_no_improve = 0

        # Keep best in RAM
        best_state = {
            "model": {k: v.detach().cpu() for k, v in trainer.model.state_dict().items()},
            "optimizer": trainer.optimizer.state_dict(),
            "epoch": epoch,
            "best_val_loss": best_val_loss,
        }

        # Save best to Drive (persistent across runtime disconnects)
        torch.save(
            {
                "epoch": epoch,
                "best_val_loss": best_val_loss,
                "model_state_dict": trainer.model.state_dict(),
                "optimizer_state_dict": trainer.optimizer.state_dict(),
                "class_to_idx": getattr(trainer.train_dataset, "class_to_idx", None),
            },
            ckpt_path,
        )

        print(f"✔ New best model at epoch {epoch+1} | saved to: {ckpt_path}")
    else:
        epochs_no_improve += 1
        print(f"No improvement ({epochs_no_improve}/{patience})")

    if epochs_no_improve >= patience:
        print("Early stopping triggered")
        break

print("Finished Training")

# Restore best model for immediate evaluation in this session
trainer.model.load_state_dict(best_state["model"])
trainer.model.to(trainer.device)
trainer.model.eval()

print(
    f"Restored best model from epoch {best_state['epoch'] + 1} "
    f"(best_val_loss={best_state['best_val_loss']:.4f})"
)
print(f"Best checkpoint saved at: {ckpt_path}")


Epoch 1


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipython-input-288/2969428516.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.use_amp):


Training loss: 0.2005
Validation loss: 0.2970
Validation acc: 87.50%
✔ New best model at epoch 1 | saved to: /content/drive/MyDrive/chest_xray/checkpoints/eva_x_tiny_binary_best.pt

Epoch 2
Training loss: 0.0628
Validation loss: 0.2698
Validation acc: 87.50%
✔ New best model at epoch 2 | saved to: /content/drive/MyDrive/chest_xray/checkpoints/eva_x_tiny_binary_best.pt

Epoch 3
Training loss: 0.0511
Validation loss: 0.1178
Validation acc: 93.75%
✔ New best model at epoch 3 | saved to: /content/drive/MyDrive/chest_xray/checkpoints/eva_x_tiny_binary_best.pt

Epoch 4
Training loss: 0.0462
Validation loss: 0.0997
Validation acc: 93.75%
✔ New best model at epoch 4 | saved to: /content/drive/MyDrive/chest_xray/checkpoints/eva_x_tiny_binary_best.pt

Epoch 5
Training loss: 0.0421
Validation loss: 0.1867
Validation acc: 87.50%
No improvement (1/4)

Epoch 6
Training loss: 0.0355
Validation loss: 0.2100
Validation acc: 87.50%
No improvement (2/4)

Epoch 7
Training loss: 0.0384
Validation loss: 0.0

KeyboardInterrupt: 

In [ ]:
'''
ckpt_path = "/content/drive/MyDrive/chest_xray/checkpoints/eva_x_tiny_binary_best.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")

model = Eva_X_Model()
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
optimizer.load_state_dict(ckpt["optimizer_state_dict"])

print("Restored epoch:", ckpt["epoch"] + 1, "best_val_loss:", ckpt["best_val_loss"])
print("class_to_idx:", ckpt.get("class_to_idx"))
'''

'\nckpt_path = "/content/drive/MyDrive/chest_xray/checkpoints/eva_x_tiny_binary_best.pt"\nckpt = torch.load(ckpt_path, map_location="cpu")\n\nmodel = Eva_X_Model()\nmodel.load_state_dict(ckpt["model_state_dict"])\nmodel.to(device)\n\noptimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)\noptimizer.load_state_dict(ckpt["optimizer_state_dict"])\n\nprint("Restored epoch:", ckpt["epoch"] + 1, "best_val_loss:", ckpt["best_val_loss"])\nprint("class_to_idx:", ckpt.get("class_to_idx"))\n'

https://huggingface.co/StanfordAIMI/CheXagent-2-3b

# Task
Add the necessary `torch` and `eva_x` imports and redefine the `Eva_X_Model` class within the code cell that loads the model checkpoint and optimizer, then print the epoch and validation loss from the loaded checkpoint.

## Add Imports and Redefine Model

### Subtask:
Add the necessary `torch` and `eva_x` imports, and re-define the `Eva_X_Model` class directly within the selected cell to ensure `eva_x_tiny_patch16` is in scope when the model is instantiated.


**Reasoning**:
To resolve the `NameError` for `eva_x_tiny_patch16` and ensure the model can be correctly instantiated, I need to add the necessary imports and redefine the `Eva_X_Model` class within the cell before attempting to load the model checkpoint.



**Reasoning**:
The previous code failed because the `device` variable was not defined in the cell where the model was being loaded. I need to define `device` before it's used to move the model to the appropriate computational device.



In [ ]:
import torch
import eva_x
from eva_x import eva_x_tiny_patch16
import torch.nn as nn
import torch.optim as optim
from re import X
import timm
import torch.nn.functional as F

class Eva_X_Model(nn.Module):
    def __init__(self):
        super().__init__()
        'img_size = 224, patch_size = 16, embed_dim = 192, depth = 12 transformer layers, num_heads = 3'

        self.eva = eva_x_tiny_patch16(pretrained="/content/drive/MyDrive/chest_xray/eva_x_tiny_patch16_merged520k_mim.pt")
        self.eva.head = nn.Linear(192, 1) #normal vs. abnormal

        #freeze entire Eva
        for p in self.eva.parameters():
            p.requires_grad = False

        #unfreeze last transformer block + norm
        for name, p in self.eva.named_parameters():
            if "head" in name or "blocks.11" in name or "norm" in name or "fc_norm" in name:
                p.requires_grad = True

    def forward(self, x):
        binary_logits = self.eva(x) #returns (B, 1) logits

        return binary_logits

ckpt_path = "/content/drive/MyDrive/chest_xray/checkpoints/eva_x_tiny_binary_best.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Define device here

model = Eva_X_Model()
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
optimizer.load_state_dict(ckpt["optimizer_state_dict"])

print("Restored epoch:", ckpt["epoch"] + 1, "best_val_loss:", ckpt["best_val_loss"])
print("class_to_idx:", ckpt.get("class_to_idx"))


In [ ]:
from PIL import Image
import torch
import torchvision.transforms as transforms
import requests
from io import BytesIO

inference_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

def predict_image(source, model, device, threshold=0.5):
    model.eval()

    # Load image from URL or local path
    if source.startswith("http"):
        response = requests.get(source)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(source).convert("RGB")

    image_tensor = inference_transforms(image)
    image_tensor = image_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image_tensor)
        prob = torch.sigmoid(logits).item()

    prediction = int(prob > threshold)

    return {
        "source": source,
        "p_abnormal": round(prob, 4),
        "prediction": prediction,
        "label_name": "abnormal" if prediction == 1 else "normal"
    }

In [ ]:
#https://radiopaedia.org/cases/pneumonia-7

url = "https://prod-images-static.radiopaedia.org/images/554213/27e6e725de1f349479fd214944aaf0d32523ec38f111107b6f347a3530e4b4ff_big_gallery.jpeg"

result = predict_image(url, model, device)
print(result)

{'source': 'https://prod-images-static.radiopaedia.org/images/554213/27e6e725de1f349479fd214944aaf0d32523ec38f111107b6f347a3530e4b4ff_big_gallery.jpeg', 'p_abnormal': 0.9988, 'prediction': 1, 'label_name': 'abnormal'}


In [ ]:
#https://radiopaedia.org/cases/pneumonia-7

url = "https://prod-images-static.radiopaedia.org/images/554213/27e6e725de1f349479fd214944aaf0d32523ec38f111107b6f347a3530e4b4ff_big_gallery.jpeg"

result = predict_image(url, model, device)
print(result)

{'source': 'https://prod-images-static.radiopaedia.org/images/554213/27e6e725de1f349479fd214944aaf0d32523ec38f111107b6f347a3530e4b4ff_big_gallery.jpeg', 'p_abnormal': 0.9988, 'prediction': 1, 'label_name': 'abnormal'}


In [ ]:
#https://radiopaedia.org/cases/normal-chest-x-ray

url = "https://prod-images-static.radiopaedia.org/images/220869/76052f7902246ff862f52f5d3cd9cd_big_gallery.jpg"

result = predict_image(url, model, device)
print(result)

{'source': 'https://prod-images-static.radiopaedia.org/images/220869/76052f7902246ff862f52f5d3cd9cd_big_gallery.jpg', 'p_abnormal': 0.0865, 'prediction': 0, 'label_name': 'normal'}


In [ ]:
#https://www.physio-pedia.com/File:Bronchitis2.png

url = "https://www.physio-pedia.com/images/e/e9/Bronchitis2.png?20170913201116"

result = predict_image(url, model, device)
print(result)

{'source': 'https://www.physio-pedia.com/images/e/e9/Bronchitis2.png?20170913201116', 'p_abnormal': 0.9405, 'prediction': 1, 'label_name': 'abnormal'}


In [ ]:
#https://www.radiology.ca/article/immigration-chest-x-ray-tuberculosis/

url = "https://www.radiology.ca/wp-content/uploads/2022/06/TB_web.jpg"

result = predict_image(url, model, device)
print(result)

{'source': 'https://www.radiology.ca/wp-content/uploads/2022/06/TB_web.jpg', 'p_abnormal': 0.9986, 'prediction': 1, 'label_name': 'abnormal'}


In [ ]:
#https://medschool.cuanschutz.edu/surgery/divisions-centers-affiliates/cardiothoracic/patient-care/lung-cancer-staging

url = "https://medschool.cuanschutz.edu/images/librariesprovider74/cardiothoracic-surgery/lung-cancer.jpg?sfvrsn=702769b9_0"

result = predict_image(url, model, device)
print(result)

{'source': 'https://medschool.cuanschutz.edu/images/librariesprovider74/cardiothoracic-surgery/lung-cancer.jpg?sfvrsn=702769b9_0', 'p_abnormal': 0.949, 'prediction': 1, 'label_name': 'abnormal'}


In [ ]:
#https://visualsonline.cancer.gov/details.cfm?imageid=2343

url = "https://visualsonline.cancer.gov/images/2343-preview.jpg"

result = predict_image(url, model, device)
print(result)

{'source': 'https://visualsonline.cancer.gov/images/2343-preview.jpg', 'p_abnormal': 0.9981, 'prediction': 1, 'label_name': 'abnormal'}


In [ ]:
#https://radiologyblog.cincinnatichildrens.org/what-can-a-chest-x-ray-show/
#https://glassboxmedicine.com/2019/02/10/radiology-normal-chest-x-rays/
#https://media.gettyimages.com/id/486466717/photo/x-ray-of-normal-healthy-chest-front-view-36-year-old-male.jpg?s=612x612&w=gi&k=20&c=ykBQMYqv0YKNtBZsOO2IfMV6p-RbaKU_bvt8qCV_4fU=

#incorrect results ...

#url = "https://glassboxmedicine.com/wp-content/uploads/2019/02/normal_cxr-heart-dia-clav.jpg"
#url = "https://media.gettyimages.com/id/486466717/photo/x-ray-of-normal-healthy-chest-front-view-36-year-old-male.jpg?s=612x612&w=gi&k=20&c=ykBQMYqv0YKNtBZsOO2IfMV6p-RbaKU_bvt8qCV_4fU="

#this one is correct
#https://www.itnonline.com/sites/default/files/styles/content_large/public/GettyImages-1140483100.jpg?itok=H5HOE8kn
url = "https://www.itnonline.com/sites/default/files/styles/content_large/public/GettyImages-1140483100.jpg?itok=H5HOE8kn"
result = predict_image(url, model, device)
print(result)

{'source': 'https://www.itnonline.com/sites/default/files/styles/content_large/public/GettyImages-1140483100.jpg?itok=H5HOE8kn', 'p_abnormal': 0.4394, 'prediction': 0, 'label_name': 'normal'}


In [ ]:
# Model: https://huggingface.co/StanfordAIMI/CheXagent-2-3b

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

CheX = "StanfordAIMI/CheXagent-2-3b"
device = "cuda" if torch.cuda.is_available() else "cpu"

CheX_tokenizer = AutoTokenizer.from_pretrained(CheX, trust_remote_code=True)

CheX_model = AutoModelForCausalLM.from_pretrained(
    CheX,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,   # or torch.float16 if bf16 not supported
    device_map={"": 0}            # forces everything onto GPU 0 (no meta shards)
)
CheX_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: Fut

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

RuntimeError: Cannot access accelerator device when none is available.

https://github.com/Stanford-AIMI/CheXagent/blob/main/model_chexagent/chexagent.py

In [ ]:
'''
import chexagent

local_chexagent = chexagent.CheXagent()

# Task 1: View Classification
path = "https://prod-images-static.radiopaedia.org/images/23511538/8a28003cc78f3549ac9f436dfe7dad_big_gallery.jpeg"
response = local_chexagent.view_classification(path)
print(f'=' * 42)
print(f'[Task 1: View Classification]')
print(f'Image: {path}')
print(f'Result: {response}')
print(f'=' * 42)
'''

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:696: UserWarning: for logit_scale: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:696: UserWarning: for logit_bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresp

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

RuntimeError: You can't move a model that has some modules offloaded to cpu or disk.

https://huggingface.co/StanfordAIMI/CheXagent-2-3b-srrg-findings#:~:text=eval()%20%23%20step%203:%20Download,Image%2Dto%2DText

In [ ]:
import tempfile
import io
import requests
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer
import tempfile

# step 1: Setup constants
CheX_model_name = "StanfordAIMI/CheXagent-2-3b-srrg-findings"
dtype = torch.bfloat16
device = "cuda" if torch.cuda.is_available() else "cpu" # Dynamically set device based on CUDA availability

# step 2: Load Processor and Model
CheX_tokenizer = AutoTokenizer.from_pretrained(CheX_model_name, trust_remote_code=True)
CheX_model = AutoModelForCausalLM.from_pretrained(
    CheX_model_name,
    trust_remote_code=True,
    torch_dtype=dtype if device == "cuda" else torch.float32, # Use bfloat16 only if CUDA is available
    device_map=device, # Forces everything onto the specified device
    low_cpu_mem_usage=False # Explicitly prevent meta loading/offloading
)
# Removed CheX_model = CheX_model.to(dtype) as it's handled by device_map and torch_dtype
CheX_model.eval()

# step 3: Download image from URL, save to a local file, and prepare path list
url = "https://huggingface.co/IAMJB/interpret-cxr-impression-baseline/resolve/main/effusions-bibasal.jpg"
resp = requests.get(url)
resp.raise_for_status()

# Use a NamedTemporaryFile so it lives on disk
with tempfile.NamedTemporaryFile(delete=False, suffix=".jpg") as tmpfile:
    tmpfile.write(resp.content)
    local_path = tmpfile.name  # this is a real file path on disk

paths = [local_path]

prompt = "Structured Radiology Report Generation for Findings Section"
# build the multimodal input
query = CheX_tokenizer.from_list_format(
    [*([{"image": img} for img in paths]), {"text": prompt}]
)

# format as a chat conversation
conv = [
    {"from": "system", "value": "You are a helpful assistant."},
    {"from": "human", "value": query},
]

# tokenize and generate
input_ids = CheX_tokenizer.apply_chat_template(
    conv, add_generation_prompt=True, return_tensors="pt"
)
output = CheX_model.generate(
    input_ids.to(device),
    do_sample=False,
    num_beams=1,
    temperature=1.0,
    top_p=1.0,
    use_cache=True,
    max_new_tokens=512,
)[0]

# decode the “findings” text
response = CheX_tokenizer.decode(output[input_ids.size(1) : -1])
print(response)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_chexagent.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/StanfordAIMI/CheXagent-2-3b-srrg-findings:
- tokenization_chexagent.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/StanfordAIMI/CheXagent-2-3b-srrg-findings:
- configuration_chexagent.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_chexagent.py: 0.00B [00:00, ?B/s]

modeling_visual.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/StanfordAIMI/CheXagent-2-3b-srrg-findings:
- modeling_visual.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/StanfordAIMI/CheXagent-2-3b-srrg-findings:
- modeling_chexagent.py
- modeling_visual.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


ValueError: Passing along a `device_map` requires `low_cpu_mem_usage=True`

In [ ]:
#https://radiopaedia.org/cases/pneumonia-7

url = "https://huggingface.co/IAMJB/interpret-cxr-impression-baseline/resolve/main/effusions-bibasal.jpg"

result = predict_image(url, model, device)
print(result)

In [ ]:
# Re-define load_image here, as it was in a deleted cell and is needed by diagnose_chest_xray
import requests
from io import BytesIO
from PIL import Image

def load_image(image_file):
    if image_file.startswith('http') or image_file.startswith('https'):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        image = Image.open(image_file).convert('RGB')
    return image

# Re-define diagnose_chest_xray with the necessary fixes.
# This version will shadow the one defined in GyrE2Dn2CTDA for this cell's execution.
# It implicitly uses `predict_image` (from Sv9tm4zKyWJ-), `CheX_tokenizer` and `CheX_model` (from 2N-R7zPv4Nmo)
# and `device` (from 579e4239) which are assumed to be available from previous executions.
def diagnose_chest_xray(image_path, eva_model, device, threshold=0.5):
    result = predict_image(image_path, eva_model, device) # `predict_image` is global

    # Build base output (always returned)
    out = {
        "source": image_path,
        "p_abnormal": result["p_abnormal"],
        "y_pred": int(result["p_abnormal"] > threshold),
        "reasoning": None
    }

    # If predicted normal, stop early
    if out["p_abnormal"] <= threshold:
        out["reasoning"] = "Classifier predicts NORMAL; reasoning model not invoked."
        print(out["reasoning"]) # Explicitly print this message
        return None # Return None for normal prediction, consistent with previous output for normal case

    # Otherwise call CheXagent
    prompt = f"""A pretrained vision transformer fine-tuned for binary classification of chest X-rays
(normal vs pneumonia-like abnormality) assigned this image an abnormal probability of
{out['p_abnormal']:.2f}. This classifier does not distinguish among specific pathologies.

Please analyze the image and describe possible radiologic findings that could explain this abnormal signal.
Provide a cautious interpretation and discuss uncertainty.
"""
    # FIX 1: Pass image_path string directly to from_list_format
    query = CheX_tokenizer.from_list_format([{'image': image_path}, {'text': prompt}])

    # conv_templates is implicitly handled by `CheX_tokenizer.apply_chat_template` if `trust_remote_code=True` was used.
    conv = [{"from": "system", "value": "You are a helpful assistant."}, {"from": "human", "value": query}]
    input_ids = CheX_tokenizer.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt")

    # FIX 2: Use CheX_model for generation, not the 'eva_model' passed as an argument.
    output = CheX_model.generate(
        input_ids.to(device), do_sample=False, num_beams=1, temperature=1., top_p=1., use_cache=True,
        max_new_tokens=512
    )[0]
    response = CheX_tokenizer.decode(output[input_ids.size(1):-1])

    print(response) # Explicitly print the LLM response as per user feedback

    return response # Return the LLM response string, as per previous behavior and user feedback.


# Original code in the selected cell
path = "/content/drive/MyDrive/chest_xray/train/PNEUMONIA/person1000_bacteria_2931.jpeg"

# Call the locally redefined diagnose_chest_xray function
result = diagnose_chest_xray(path, model, device)
# The result variable will now hold the LLM response string (or None), which was already printed inside the function.
# Printing 'result' again will show the same string.
print(result)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


NotImplementedError: Cannot copy out of meta tensor; no data!

In [ ]:
#url = "https://visualsonline.cancer.gov/images/2343-preview.jpg"

path = "/content/drive/MyDrive/chest_xray/train/NORMAL/IM-0115-0001.jpeg"

result = diagnose_chest_xray(path, model, device)
print(result)

Classifier predicts NORMAL; reasoning model not invoked.
None


In [ ]:
#url = "https://visualsonline.cancer.gov/images/2343-preview.jpg"

path = "/content/drive/MyDrive/chest_xray/train/PNEUMONIA/person1000_bacteria_2931.jpeg"

result = diagnose_chest_xray(path, model, device)
print(result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


A pretrained vision transformer fine-tuned for binary classification of chest X-rays 
(normal vs pneumonia-like abnormality) assigned this image an abnormal probability of 
1.00. This classifier does not distinguish among specific pathologies.

Please analyze the image and describe possible radiologic findings that could explain this abnormal signal.
Provide a cautious interpretation and discuss uncertainty.



In [ ]:
url = "https://www.radiology.ca/wp-content/uploads/2022/06/TB_web.jpg"

result = diagnose_chest_xray(url, model, device)
print(result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


A pretrained vision transformer fine-tuned for binary classification of chest X-rays 
(normal vs pneumonia-like abnormality) assigned this image an abnormal probability of 
1.00. This classifier does not distinguish among specific pathologies.

Please analyze the image and describe possible radiologic findings that could explain this abnormal signal.
Provide a cautious interpretation and discuss uncertainty.

None


In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score
import numpy as np
import torch

def evaluate_full(model, loader, device, name="Eval"):
    model.eval()

    all_labels = []
    all_probs = []
    all_preds = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            probs = torch.sigmoid(logits)

            preds = (probs > 0.5).int().squeeze(1)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.squeeze(1).cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    # Accuracy
    acc = (all_preds == all_labels).mean()

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()

    # Sensitivity & Specificity
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    # AUROC
    auc = roc_auc_score(all_labels, all_probs)

    print(f"\n{name} Results:")
    print(f"Accuracy: {acc*100:.2f}%")
    print(f"Sensitivity (Recall Pneumonia): {sensitivity:.4f}")
    print(f"Specificity (Recall Normal): {specificity:.4f}")
    print(f"AUROC: {auc:.4f}")
    print("\nConfusion Matrix:")
    print(f"TN: {tn}  FP: {fp}")
    print(f"FN: {fn}  TP: {tp}")

    return {
        "accuracy": acc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "auc": auc,
        "confusion_matrix": (tn, fp, fn, tp)
    }

evaluate_full(model, train_dataloader, device, "Eval")
evaluate_full(model, val_dataloader, device, "Eval")
evaluate_full(model, test_dataloader, device, "Eval")